# Advanced Audio Deepfake Detection (ASVspoof)

In [1]:
!pip install librosa tensorflow numpy pandas matplotlib scikit-learn tqdm

In [2]:
import os
import numpy as np
import pandas as pd
import librosa
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras import layers, models


In [12]:
label_path = "../audio/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt"
cols = ['speaker','file','env','attack','label']
df = pd.read_csv(label_path, sep=' ', names=cols)
df.head()

,speaker,file,env,attack,label
0,LA_0079,LA_T_1138215,-,-,bonafide
1,LA_0079,LA_T_1271820,-,-,bonafide
2,LA_0079,LA_T_1272637,-,-,bonafide
3,LA_0079,LA_T_1276960,-,-,bonafide
4,LA_0079,LA_T_1341447,-,-,bonafide


In [13]:
def augment_audio(audio, sr):
    noise = np.random.randn(len(audio))
    audio_noise = audio + 0.005 * noise
    shift = np.roll(audio, int(sr * 0.2))
    return [audio, audio_noise, shift]

In [15]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

[]


In [14]:
X, y = [], []
base_path = '../audio/LA/ASVspoof2019_LA_train/flac/'

for _, row in tqdm(df.iterrows(), total=5000):
    file_name = row['file'] + '.flac'
    label = 1 if row['label']=='spoof' else 0
    file_path = os.path.join(base_path, file_name)

    if os.path.exists(file_path):
        try:
            audio, sr = librosa.load(file_path, sr=16000, duration=4)
            audios = augment_audio(audio, sr)

            for a in audios:
                spec = librosa.feature.melspectrogram(y=a, sr=sr, n_mels=128)
                log_spec = librosa.power_to_db(spec)
                X.append(log_spec)
                y.append(label)
        except:
            continue

X = np.array(X)[..., np.newaxis]
y = np.array(y)
print(X.shape)

25380it [2:08:56,  3.28it/s]                                                                                           


ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 2 dimensions. The detected shape was (76140, 128) + inhomogeneous part.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
model = models.Sequential([
 layers.Conv2D(32,(3,3),activation='relu',input_shape=X.shape[1:]),
 layers.BatchNormalization(),
 layers.MaxPooling2D(2,2),
 layers.Conv2D(64,(3,3),activation='relu'),
 layers.BatchNormalization(),
 layers.MaxPooling2D(2,2),
 layers.Conv2D(128,(3,3),activation='relu'),
 layers.BatchNormalization(),
 layers.MaxPooling2D(2,2),
 layers.GlobalAveragePooling2D(),
 layers.Dense(128,activation='relu'),
 layers.Dropout(0.4),
 layers.Dense(1,activation='sigmoid')
])

model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
model.summary()

In [ ]:
callbacks=[
 tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
 tf.keras.callbacks.ReduceLROnPlateau(patience=2)
]

In [ ]:
history=model.fit(X_train,y_train,epochs=10,batch_size=32,validation_data=(X_test,y_test),callbacks=callbacks)

In [ ]:
y_pred=(model.predict(X_test)>0.5).astype('int32')
print(classification_report(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))

In [ ]:
model.save('advanced_audio_model.h5')